In [ ]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel,  AutoModelForSeq2SeqLM
import pandas as pd
import numpy as np

In [ ]:
from tqdm import tqdm
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

In [ ]:
from sentence_transformers import SentenceTransformer, util
paraph_model = SentenceTransformer('sentence-transformers/paraphrase-mpnet-base-v2').to(device)
#модель, на выходе которой эмбеддинг предложения, с косинусным сходством близкому к 1 для предложений похожих по смыслу


In [ ]:
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)

EMB_MATRIX = bert_model.embeddings.word_embeddings.weight

In [ ]:
!pip install lightning==2.4.0

In [ ]:
import lightning as L

In [ ]:
path = "/content/"
data_cnbc = pd.read_csv(path + "cnbc_headlines.csv").dropna()
data_guar =  pd.read_csv(path + "guardian_headlines.csv").dropna()
data_reut = pd.read_csv(path+ "reuters_headlines.csv").dropna()
data = pd.concat([data_reut['Headlines'],data_cnbc['Headlines'],data_guar['Headlines']], ignore_index=True)#,data_reut['Description'],data_cnbc['Description']])

In [ ]:
from sklearn.model_selection import train_test_split
train_texts, test_texts = train_test_split(data, test_size=0.1, random_state=42)

In [ ]:
class FinancialNewsDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        if isinstance(text, str):
            return text
        else:
            return [text.iloc[i].lstrip("paraphrasedoutput: ") for i in range(6)]

train_data = FinancialNewsDataset(train_texts)
test_data = FinancialNewsDataset(test_texts)
len(train_texts), len(test_texts)

(48033, 5337)

In [ ]:
train_ph = pd.read_csv(path+"paraphrased.csv")

In [ ]:
train_paraph_data = FinancialNewsDataset(train_ph)

In [ ]:
def train_collate_fn(
    tokenizer: AutoTokenizer, batch: list[str]
) -> tuple[Tensor, Tensor]:
    res = []
    for b in zip(*batch):
      encoded_batch = tokenizer(
          b, padding="longest", return_tensors="pt", return_token_type_ids=False).to(device)
      res.append(encoded_batch)
    return res

len(tokenizer)

30873

In [ ]:
ph_loader = DataLoader(train_paraph_data, batch_size=64, shuffle=False, collate_fn=lambda batch:train_collate_fn(tokenizer,batch))
b = next(iter(ph_loader))

# Модели + обучение

In [ ]:
def collate_fn(
    tokenizer: AutoTokenizer, batch: list[str]
) -> tuple[Tensor, Tensor]:
    encoded_batch = tokenizer(
        batch, padding="longest", return_tensors="pt", return_token_type_ids=False)
    return encoded_batch.to(device)

len(tokenizer)

30873

In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=False, collate_fn=lambda batch:collate_fn(tokenizer,batch))
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=lambda batch:collate_fn(tokenizer,batch))
batch = next(iter(test_loader))
input_ids, attention_mask = batch['input_ids'], batch['attention_mask']
input_ids.shape

torch.Size([64, 29])

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, output_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim),
            #nn.Dropout(p=0.01)
        )

    def forward(self, x, mask=None):
        attn_scores = self.attn(x).squeeze(-1)  # (B, T)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)  # (B, T)
        return torch.sum(attn_weights.unsqueeze(-1) * self.proj(x), dim=1)  # (B, H)


at = AttentionPooling(768, 1024).to(device)
x= bert_model.cuda()(input_ids).last_hidden_state
at(x).shape

torch.Size([64, 1024])

In [ ]:
class FactorizedOutput(nn.Module):
    def __init__(self, hidden_dim, vocab_size, factor_dim):
        super(FactorizedOutput, self).__init__()
        self.dropout = nn.Dropout(0.5)
        self.proj_down = nn.Linear(hidden_dim, factor_dim, bias=False)
        self.proj_up = nn.Linear(factor_dim, vocab_size, bias=False)

    def forward(self, x):
        # Down-projection
        x = self.proj_down(x)  # (B, T, factor_dim)
        # Up-projection to vocab size
        logits = self.proj_up(self.dropout(x))  # (B, T, vocab_size)
        return logits

In [ ]:
import math


class EncoderDecoderModel(nn.Module):
    def __init__(self, bert_model, hidden_dim=768, num_layers=2, nhead=8, max_length=1000, vocab_size=len(tokenizer),dropout=1e-4, sent_dim=768, tokenizer=tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.bert = bert_model  # ЭНКОДЕР (BERT)
        self.bert.requires_grad_(False)
        self.hidden_dim = hidden_dim
        self.max_length = max_length
        self.vocab_size = vocab_size
        self.sent_dim = sent_dim


        self.attn_pool = AttentionPooling(hidden_dim, sent_dim)


        self.tgt_projector= nn.Sequential(
            nn.Linear(hidden_dim, sent_dim),
            nn.LayerNorm(sent_dim),
            nn.GELU(),
            #nn.Dropout(p=0.01)
        )

        decoder_layer = nn.TransformerDecoderLayer(d_model=sent_dim, nhead=nhead,dim_feedforward=sent_dim,dropout=dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Выходной слой для предсказания токенов
        self.fc_out = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(sent_dim, vocab_size)
        )
        #self.factorized_output = FactorizedOutput(sent_dim, vocab_size, sent_dim//2)

        # Генерация синусоидальных позиционных эмбеддингов
        self.register_buffer("positional_encoding", self.sinusoidal_positional_encoding(max_length, sent_dim))

    def sinusoidal_positional_encoding(self, seq_length, hidden_dim):
        position = torch.arange(seq_length).unsqueeze(1).float()  # (T, 1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).float() * (-math.log(10000.0) / hidden_dim))

        pe = torch.zeros(seq_length, hidden_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0).to(device)  # (1, T, H)

    def get_sentence_emb(self, input_ids, attention_mask):
        encoded = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state # (B, T, H)
        sent_embedding = self.attn_pool(encoded, attention_mask)
        return sent_embedding

    def get_logits(self, tgt_ids, sent_emb, attention_mask):
        batch_size, seq_length = tgt_ids.shape  # (B, T)

        tgt = self.bert.embeddings.word_embeddings(tgt_ids)



        memory = sent_emb.unsqueeze(1)#torch.stack(memory, dim=1)



        # Декодерные входы (позиционные эмбеддинги)
        tgt = self.tgt_projector(tgt)

        # Добавляем синусоидальные позиции
        tgt = tgt + self.positional_encoding[:, :seq_length, :]  # (B, T, S)

        # ПРОГОН ЧЕРЕЗ ДЕКОДЕР

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_length).to(device)

        decoder_output = self.transformer_decoder(tgt,
                                                  memory=memory,
                                                  tgt_mask=tgt_mask.bool(),
                                                  tgt_key_padding_mask=(attention_mask==0)
                                                  )  # (B, T, S)



        # ПРОГОН ЧЕРЕЗ ВЫХОДНОЙ ЛИНЕЙНЫЙ СЛОЙ
        logits = self.fc_out(decoder_output)  # (B, T, V)

        return logits

    def forward(self, input_ids, attention_mask, tgt_ids=None):
        batch_size, seq_length = input_ids.shape  # (B, T)


        sent_emb = self.get_sentence_emb(input_ids, attention_mask)  # (B, S)
        if tgt_ids is None:
          tgt_ids = input_ids



        token_logits = self.get_logits(tgt_ids, sent_emb, attention_mask)  # (B, T, V)

        return token_logits, sent_emb  # (B, T, V)

    @torch.inference_mode()
    def generate_from_embedding(self, sent_emb: torch.Tensor, max_len: int = 100):
        self.eval()

        B, S = sent_emb.shape

        memory = sent_emb.unsqueeze(1)

        tgt_ids = torch.full((B, 1), self.tokenizer.cls_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_len):
            t = tgt_ids.size(1)

            pos_emb = self.positional_encoding[:, :t, :]

            tgt = self.bert.embeddings.word_embeddings(tgt_ids)

            tgt = self.tgt_projector(tgt) + pos_emb

            tgt_mask = nn.Transformer.generate_square_subsequent_mask(t).to(device)

            decoder_output = self.transformer_decoder(tgt,
                                                      memory=memory,
                                                      tgt_mask=tgt_mask.bool(),
                                                      )
            logits = self.fc_out(decoder_output[:,-1,:])

            next_token = torch.argmax(logits, dim=-1, keepdim=True)

            tgt_ids = torch.cat([tgt_ids, next_token], dim=-1)


            finished = finished | (next_token.squeeze(1) == self.tokenizer.sep_token_id)
            if finished.all():
                break


        return self.tokenizer.batch_decode(tgt_ids, skip_special_tokens=True)



In [ ]:
class LightningAutoencoder(L.LightningModule):
    def __init__(self, model, learning_rate=1e-4, tokenizer=tokenizer):
        super().__init__()
        self.model = model
        self.criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
        self.learning_rate = learning_rate
        self.tokenizer = tokenizer

    def get_sentence_emb(self, batch):
        return self.model.get_sentence_emb(batch['input_ids'], batch['attention_mask'])

    def forward(self, batch):
        return self.model(batch['input_ids'], batch['attention_mask'])

    def training_step(self, batch, batch_idx):
        input_ids, attention_mask = batch['input_ids'], batch['attention_mask']

        outputs, sent_emb = self.model(input_ids, attention_mask)
        outputs = outputs[:,:-1,:]
        targets = input_ids[:, 1:]


        ce_loss = self.criterion(outputs.reshape(-1, self.model.vocab_size), targets.reshape(-1))

        loss = ce_loss
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

        accuracy = torch.logical_and(targets == outputs.argmax(dim=-1), attention_mask[:, 1:]).sum() / attention_mask[:, 1:].sum()
        self.log("train_accuracy", accuracy.item(), on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids, attention_mask = batch['input_ids'], batch['attention_mask']



        outputs, sent_emb = self.model(input_ids, attention_mask)

        outputs = outputs[:,:-1,:]

        targets = input_ids[:, 1:].contiguous()


        ce_loss = self.criterion(outputs.reshape(-1, self.model.vocab_size), targets.reshape(-1))

        loss = ce_loss
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

        accuracy = torch.logical_and(targets == outputs.argmax(dim=-1), attention_mask[:, 1:]).sum() / attention_mask.sum()
        self.log("val_accuracy", accuracy.item(), on_step=False, on_epoch=True, prog_bar=True)



        output_text = self.generate_from_embedding(sent_emb, max_len=100)
        input_text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)


        in_embeddings = paraph_model.encode(input_text, convert_to_tensor=True)
        out_embeddings = paraph_model.encode(output_text, convert_to_tensor=True)

        ph = torch.diagonal(util.pytorch_cos_sim(in_embeddings, out_embeddings))
        self.log("ph_sim", ph.mean(), on_step=False, on_epoch=True, prog_bar=True)
        return {
            "loss": loss,
            "preds": outputs,
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=0.01)
        # давайте кроме оптимизатора создадим ещё расписание для шага оптимизации
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.1, min_lr=1e-5,
                patience=10
            ),
            "monitor": "ph_sim"
        }


    @torch.inference_mode()
    def generate_from_embedding(self, sent_emb: torch.Tensor, max_len: int = 100):
        self.eval()
        return self.model.generate_from_embedding(sent_emb, max_len)


In [ ]:
def top_k(batch, k=5):
    attention_mask = batch['attention_mask']
    input_ids = batch['input_ids']
    logits,_  = model.model(input_ids, attention_mask)  # Замените B, T, V на соответствующие значения
    logits = logits[:,:-1,:]
    targets = input_ids[:,1:]  # Целевые индексы токенов

    # Получаем топ-5 предсказаний для каждого токена
    topk_probs, topk_indices = torch.topk(logits, k, dim=-1)  # top5_indices имеет форму [B, T, 5]

    # Сравниваем целевые токены с топ-5 предсказаниями
    # Добавляем дополнительное измерение к targets для сравнения
    targets_expanded = targets.unsqueeze(-1)  # Форма: [B, T, 1]

    # Проверяем, совпадает ли целевой токен с одним из топ-5 предсказаний
    correct = (topk_indices == targets_expanded).any(dim=-1)  # Форма: [B, T], значения True/False

    # Вычисляем точность: среднее значение по всем токенам
    topk_accuracy = torch.logical_and(correct, attention_mask[:, 1:]).sum() / attention_mask.sum()

    #print(f"Top-5 Accuracy: {top5_accuracy:.4f}")
    return topk_accuracy


In [ ]:
def top_k_ph(batch, k=5):
    input_ids, attention_mask = batch[0]['input_ids'], batch[0]['attention_mask']
    pr_ids1, mask1 = batch[1]['input_ids'], batch[1]['attention_mask']
    pr_ids2, mask2 = batch[2]['input_ids'], batch[2]['attention_mask']
    pr_ids3, mask3 = batch[3]['input_ids'], batch[3]['attention_mask']
    sent_emb = model.model.get_sentence_emb(input_ids, attention_mask)
    logits,_  = model.model(input_ids, attention_mask)  # Замените B, T, V на соответствующие значения
    logits = logits[:,:-1,:]

    outputs1 = model.model.get_logits(pr_ids1, sent_emb, mask1)[:,:-1,:]
    outputs2 = model.model.get_logits(pr_ids2, sent_emb, mask2)[:,:-1,:]
    outputs3 = model.model.get_logits(pr_ids3, sent_emb, mask3)[:,:-1,:]
    targets = input_ids[:,1:]  # Целевые индексы токенов
    targets1 = pr_ids1[:,1:]
    targets2 = pr_ids2[:,1:]
    targets3 = pr_ids3[:,1:]
    # Получаем топ-5 предсказаний для каждого токена
    topk_probs, topk_indices = torch.topk(logits, k, dim=-1)  # top5_indices имеет форму [B, T, 5]
    topk_probs1, topk_indices1 = torch.topk(outputs1, k, dim=-1)
    topk_probs2, topk_indices2 = torch.topk(outputs2, k, dim=-1)
    topk_probs3, topk_indices3 = torch.topk(outputs3, k, dim=-1)
    # Сравниваем целевые токены с топ-5 предсказаниями
    # Добавляем дополнительное измерение к targets для сравнения
    targets_expanded = targets.unsqueeze(-1)  # Форма: [B, T, 1]

    # Проверяем, совпадает ли целевой токен с одним из топ-5 предсказаний
    correct = (topk_indices == targets_expanded).any(dim=-1)  # Форма: [B, T], значения True/False
    correct1 = (topk_indices1 == targets1.unsqueeze(-1)).any(dim=-1)
    correct2= (topk_indices2 == targets2.unsqueeze(-1)).any(dim=-1)
    correct3 = (topk_indices3 == targets3.unsqueeze(-1)).any(dim=-1)
    # Вычисляем точность: среднее значение по всем токенам
    topk_accuracy = torch.logical_and(correct, attention_mask[:, 1:]).sum() / attention_mask.sum()
    topk_accuracy1 = torch.logical_and(correct1, mask1[:, 1:]).sum() / mask1.sum()
    topk_accuracy2 = torch.logical_and(correct2, mask2[:, 1:]).sum() / mask2.sum()
    topk_accuracy3 = torch.logical_and(correct3, mask3[:, 1:]).sum() / mask3.sum()

    #print(f"Top-5 Accuracy: {top5_accuracy:.4f}")
    return (topk_accuracy+topk_accuracy1+topk_accuracy2+topk_accuracy3)/4


In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, sent_dim=768,max_length=1000).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)
#checkpoint_path = next(Path(f"/content/tb_logs/lightning_logs/768/checkpoints").glob("*.ckpt"))
#model.load_state_dict(torch.load("/content/tb_logs/lightning_logs/768/checkpoints/last.ckpt", weights_only=True)["state_dict"])
model.load_state_dict(torch.load('/content/model768.pth', weights_only=True))
#w = model.model.attn_pool.attn.weight

<All keys matched successfully>

In [ ]:
from lightning.pytorch.callbacks.model_summary import summarize

print(summarize(model,2))

  | Name                      | Type                | Params | Mode 
--------------------------------------------------------------------------
0 | model                     | EncoderDecoderModel | 146 M  | train
1 | model.bert                | BertModel           | 109 M  | eval 
2 | model.attn_pool           | AttentionPooling    | 592 K  | train
3 | model.tgt_projector       | Sequential          | 592 K  | train
4 | model.transformer_decoder | TransformerDecoder  | 11.8 M | train
5 | model.fc_out              | Sequential          | 23.7 M | train
6 | criterion                 | CrossEntropyLoss    | 0      | train
--------------------------------------------------------------------------
36.7 M    Trainable params
109 M     Non-trainable params
146 M     Total params
585.997   Total estimated model params size (MB)
45        Modules in train mode
228       Modules in eval mode


In [ ]:
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    ModelCheckpoint(
        filename="{epoch}-{val_loss:.2f}",
        monitor="val_loss",
        mode="min",
        save_top_k=3,
        save_last=True,
    )
]
trainer = L.Trainer(
    max_epochs=25,
    accelerator="auto",
    logger=TensorBoardLogger(save_dir="tb_logs", version='768'),
    callbacks=callbacks
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(model, train_loader, test_loader, ckpt_path="/content/tb_logs/lightning_logs/768/checkpoints/epoch=14-val_loss=1.74.ckpt")

# val_loss=2.130, val_accuracy=0.595, ph_sim=0.680, train_loss=1.240, train_accuracy=0.689
#val_loss=2.580, val_accuracy=0.527, ph_sim=0.606, train_loss=1.620, train_accuracy=0.659]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory tb_logs/lightning_logs/768/checkpoints exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/tb_logs/lightning_logs/768/checkpoints/epoch=14-val_loss=1.74.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/tb_logs/lightning_logs/768/checkpoints/epoch=14-val_loss=1.74.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type                | Params | Mode 
----------------------------------------------------------
0 | model     | EncoderDecoderModel | 146 M  | train
1 | criterion | CrossEntropyLoss    | 0      | train
----------------------------------------------------------
36.7 M    Trainable params
109 M     Non-trainable params
146 M     Total params
585.997   Total estimated model par

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=25` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=25` reached.


In [ ]:
torch.save(model.state_dict(), "/content/model768.pth")

In [ ]:
import torch
import torch.nn.functional as F

@torch.inference_mode()
def batch_beam_search_from_embedding_topk(model, sent_emb: torch.Tensor, beam_width=5, max_len=50):
    model.eval()
    device = sent_emb.device
    B = sent_emb.size(0)
    memory = sent_emb.unsqueeze(1)  # [B, 1, S]

    cls_id = model.tokenizer.cls_token_id
    sep_id = model.tokenizer.sep_token_id

    all_decoded_sequences = []

    for b in range(B):
        mem = memory[b:b+1]  # [1, 1, S]

        beams = [(torch.tensor([[cls_id]], device=device), 0.0)]  # list of (seq, score)

        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                if seq[0, -1].item() == sep_id:
                    candidates.append((seq, score))
                    continue

                t = seq.size(1)
                pos_emb = model.positional_encoding[:, :t, :]  # [1, T, D]
                tgt = model.bert.embeddings.word_embeddings(seq)  # [1, T, D]
                tgt = model.tgt_projector(tgt) + pos_emb
                tgt_mask = torch.nn.Transformer.generate_square_subsequent_mask(t).to(device)

                decoder_output = model.transformer_decoder(tgt,
                                                           memory=mem,
                                                           tgt_mask=tgt_mask.bool())  # [1, T, D]
                logits = model.fc_out(decoder_output[:, -1, :])  # [1, vocab]
                log_probs = F.log_softmax(logits, dim=-1)  # [1, vocab]

                topk_log_probs, topk_ids = torch.topk(log_probs, beam_width, dim=-1)  # [1, K]

                for k in range(beam_width):
                    next_token_id = topk_ids[0, k].unsqueeze(0).unsqueeze(0)  # [1, 1]
                    new_seq = torch.cat([seq, next_token_id], dim=1)  # [1, T+1]
                    new_score = score + topk_log_probs[0, k].item()
                    candidates.append((new_seq, new_score))

            # отбираем top-k лучших
            beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_width]

            # если все beams уже закончились — можно выйти раньше
            if all(seq[0, -1].item() == sep_id for seq, _ in beams):
                break

        # декодируем все top-k гипотез
        decoded_k = [
            model.tokenizer.decode(seq[0], skip_special_tokens=True)
            for seq, _ in beams
        ]
        all_decoded_sequences.append(decoded_k)

    return all_decoded_sequences  # list of B items, each is list of K strings


In [ ]:
def get_ph_score(output_text, input_text):
      in_embeddings = paraph_model.encode(input_text, convert_to_tensor=True)
      out_embeddings = paraph_model.encode(output_text, convert_to_tensor=True)
      return torch.diagonal(util.pytorch_cos_sim(in_embeddings, out_embeddings)).mean()

def diff_paraph_ce_with_ce(model, batch):
      input_ids, attention_mask = batch[0]['input_ids'], batch[0]['attention_mask']
      pr_ids1, mask1 = batch[1]['input_ids'], batch[1]['attention_mask']
      pr_ids2, mask2 = batch[2]['input_ids'], batch[2]['attention_mask']
      pr_ids3, mask3 = batch[3]['input_ids'], batch[3]['attention_mask']
      sent_emb = model.model.get_sentence_emb(input_ids, attention_mask)
      output_text = model.generate_from_embedding(sent_emb, max_len=100)

      input_text = model.tokenizer.batch_decode(input_ids, skip_special_tokens=True)
      pr_text1 = model.tokenizer.batch_decode(pr_ids1, skip_special_tokens=True)
      pr_text2 = model.tokenizer.batch_decode(pr_ids2, skip_special_tokens=True)
      pr_text3 = model.tokenizer.batch_decode(pr_ids3, skip_special_tokens=True)

      ph=get_ph_score(output_text, input_text)
      ph1=get_ph_score(output_text, pr_text1)
      ph2=get_ph_score(output_text, pr_text1)
      ph3=get_ph_score(output_text, pr_text1)
      return (ph+ph1+ph2+ph3)/4

In [ ]:
ph_sim=0
for b in tqdm(ph_loader):
  ph_sim+=diff_paraph_ce_with_ce(model, b)
ph_sim/=len(ph_loader)

100%|██████████| 751/751 [17:18<00:00,  1.38s/it]


In [ ]:
ph_sim

tensor(0.8296, device='cuda:0')

In [ ]:
from tqdm import tqdm

In [ ]:
top_5=0
for b in tqdm(test_loader):
  top_5+=top_k(b)
top_5/=len(test_loader)
top_5

100%|██████████| 84/84 [00:17<00:00,  4.90it/s]


tensor(0.7713, device='cuda:0')

In [ ]:
top_5=0
for b in tqdm(ph_loader):
  top_5+=top_k_ph(b)
top_5/=len(ph_loader)
top_5

100%|██████████| 751/751 [07:53<00:00,  1.59it/s]


tensor(0.5948, device='cuda:0')

In [ ]:
def beam_ph(model, batch, beam_width):
    input_text =get_input_text(batch)
    sent_emb = model.get_sentence_emb(batch)
    output_text = batch_beam_search_from_embedding_topk(model.model, sent_emb, beam_width=beam_width)
    output_text =  [out[0] for out in output_text]
    return get_ph_score(output_text, input_text)

In [ ]:
ph_beam_5=0
k=30
for b in tqdm(test_loader):
  if k == 0:
    break
  k-=1
  ph_beam_5+=beam_ph(model, b, 5)
ph_beam_5/=30
ph_beam_5

 36%|███▌      | 30/84 [08:11<14:45, 16.39s/it]


tensor(0.6874, device='cuda:0')

In [ ]:
ph_beam_5=0
k=30
for b in tqdm(test_loader):
  if k == 0:
    break
  k-=1
  ph_beam_5+=beam_ph(model, b,1)
ph_beam_5/=30
ph_beam_5

 36%|███▌      | 30/84 [01:41<03:02,  3.38s/it]


tensor(0.6655, device='cuda:0')

In [ ]:
ph_beam_5=0
k=30
for b in tqdm(test_loader):
  if k == 0:
    break
  k-=1
  ph_beam_5+=beam_ph(model, b,10)
ph_beam_5/=30
ph_beam_5

 36%|███▌      | 30/84 [16:53<30:23, 33.78s/it]


tensor(0.6894, device='cuda:0')

In [ ]:
ph_beam_5


tensor(0.6612, device='cuda:0')

In [ ]:
@torch.no_grad()
def get_ph_sent_emb(model, batch):
  input_ids, attention_mask = batch[0]['input_ids'], batch[0]['attention_mask']
  pr_ids1, mask1 = batch[1]['input_ids'], batch[1]['attention_mask']
  pr_ids2, mask2 = batch[2]['input_ids'], batch[2]['attention_mask']
  pr_ids3, mask3 = batch[3]['input_ids'], batch[3]['attention_mask']
  sent_emb = model.model.get_sentence_emb(input_ids, attention_mask)
  sent_emb1 = model.model.get_sentence_emb(pr_ids1, mask1)
  sent_emb2 = model.model.get_sentence_emb(pr_ids2, mask2)
  sent_emb3 = model.model.get_sentence_emb(pr_ids3, mask3)
  input_text = model.tokenizer.batch_decode(input_ids, skip_special_tokens=True)
  pr_text1 = model.tokenizer.batch_decode(pr_ids1, skip_special_tokens=True)
  pr_text2 = model.tokenizer.batch_decode(pr_ids2, skip_special_tokens=True)
  pr_text3 = model.tokenizer.batch_decode(pr_ids3, skip_special_tokens=True)
  return torch.stack([sent_emb,sent_emb1,sent_emb2,sent_emb3], dim=1), [input_text,pr_text1 ,pr_text2,pr_text3]


In [ ]:
b= next(iter(ph_loader))

In [ ]:
embeds, text = get_ph_sent_emb(model, b)

In [ ]:
import plotly.express as px


In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import pandas as pd

# ВАШИ ДАННЫЕ
embeddings = embeds[50:55].cpu().numpy()

# Список перефраз (замените на реальные тексты)
paraphrases_list = list(zip(*text))[50:55]


N, P, D = embeddings.shape
flat_data = embeddings.reshape(-1, D)

# PCA
pca = PCA(n_components=3)
coords_3d = pca.fit_transform(flat_data)

# Подготовка данных
hover_texts = []
sentence_colors = []

for i in range(N):
    for j in range(P):
        # Текст для всплывающей подсказки
        hover_text = (
            f"<b>📝 Предложение {i+1}</b><br>"
            f"<b>🔄 Перефраза {j+1}</b><br>"
            f"<i>Текст: {paraphrases_list[i][j]}</i><br>"
            f"<b>📊 Координаты:</b><br>"
            f"PC1: {coords_3d[i*P + j, 0]:.3f}<br>"
            f"PC2: {coords_3d[i*P + j, 1]:.3f}<br>"
            f"PC3: {coords_3d[i*P + j, 2]:.3f}"
        )
        hover_texts.append(hover_text)
        sentence_colors.append(i)

# Создаём DataFrame для удобства
df = pd.DataFrame({
    'x': coords_3d[:, 0],
    'y': coords_3d[:, 1],
    'z': coords_3d[:, 2],
    'sentence_id': sentence_colors,
    'hover_text': hover_texts
})

# График
fig = go.Figure()

colors = px.colors.qualitative.Plotly

for sentence_id in range(N):
    df_sentence = df[df['sentence_id'] == sentence_id]

    fig.add_trace(go.Scatter3d(
        x=df_sentence['x'],
        y=df_sentence['y'],
        z=df_sentence['z'],
        mode='markers',
        name=f'Предложение {sentence_id+1}',
        marker=dict(
            size=10,
            color=colors[sentence_id % len(colors)],
            opacity=0.7,
            symbol='circle',
            line=dict(width=1, color='DarkSlateGrey')
        ),
        text=df_sentence['hover_text'],
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>'
    ))

# Настройки
fig.update_layout(
    title={
        'text': '🔍 3D Визуализация эмбеддингов предложений<br>'
                '<span style="font-size:14px">Наведите/нажмите на точку для просмотра текста перефразы</span>',
        'x': 0.5,
        'xanchor': 'center'
    },
    scene=dict(
        xaxis_title='<b>PC1</b>',
        yaxis_title='<b>PC2</b>',
        zaxis_title='<b>PC3</b>',
        bgcolor='rgba(240,240,240,0.9)'
    ),
    legend=dict(
        title='<b>Предложения</b>',
        x=0.85,
        y=0.95,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='black',
        borderwidth=1
    ),
    width=1000,
    height=800
)

fig.show()

# Сохраняем
fig.write_html("embeddings_with_text.html")
print("✅ График сохранён в 'embeddings_with_text.html'")

✅ График сохранён в 'embeddings_with_text.html'


In [ ]:
model.to(device)
batch_iter = iter(test_loader)

In [ ]:
batch = next(batch_iter)

In [ ]:
batch['input_ids']=batch['input_ids'][0:10]
batch['attention_mask']= batch['attention_mask'][0:10]

In [ ]:
def get_input_text(batch):
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    input_text = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
    return  input_text

In [ ]:
def res_text(model, batch, beam=True, stoch=False):
    input_text =get_input_text(batch)
    sent_emb = model.get_sentence_emb(batch)
    if stoch:
      sent_emb+=torch.randn_like(sent_emb)
    if beam:
      output_text = batch_beam_search_from_embedding_topk(model.model, sent_emb)
    else:
      output_text = model.model.generate_from_embedding(sent_emb, max_len=1000)
    output_text =  [out[0] for out in output_text]
    return list(zip(input_text, output_text))

In [ ]:
res_text(model, batch)

[('Lumentum says halting all shipments, cuts quarterly forecast',
  'Saudi announces all halting output, says company reserves'),
 ('’ s would devastate business – the must be hoping that he ’ s lying',
  '’ s would be devastate – it must pick the that ’ s business'),
 ("court rules against in ' dieselgate ' scandal",
  "court rules against in ' scandal '"),
 ('Equity trading strength boosts profits at Morgan Stanley,',
  'Early trading profits boosts trading at Porgan Stanley,'),
 ('Switch energy providers now to escape price rises',
  'Smart energy firms now switch to price hikes'),
 ("China President says goal of and is advance ' win - win cooperation '",
  "China President is aim to win - and goal of ' success '"),
 ('minimum wage rises will put more jobs at risk of automation',
  'minimum wage rises will bring more jobs at risk'),
 ('Eni cuts 2020, 2021 capex to mitigate Coronavirus hit',
  'UniCredit cuts 2020 spending, eyes 2020 boost to coronavirus'),
 ("doubts add to ' s heada

In [ ]:
res_text(model, batch)

[('lumentum says halting all huawei shipments, cuts quarterly forecast',
  'rolls - royce says halting all chinese sales, cuts forecast'),
 ('johnson ’ s brexit would devastate business – the cbi must be hoping that he ’ s lying',
  'brexit ’ s johnson would be scrapping that ’ s government would hurt the coronavirus'),
 ("german court rules against volkswagen in'dieselgate'scandal",
  "french court rules against volkswagen'in diesel scandal '"),
 ('equity trading strength boosts profits at morgan stanley, goldman',
  'trading strength boosts at goldman sachs, boosting profit'),
 ('switch energy providers now to escape price rises',
  'switch energy providers to stop price hikes now'),
 ("china president xi says goal of belt and road is advance'win - win cooperation '",
  "china president says xi is priority of progress and aims for'long - road trade deal"),
 ('fears minimum wage rises will put more jobs at risk of automation',
  'minimum wage rises will put more risk of jobs at risk')

In [ ]:
def f(res):
  in_embeddings = paraph_model.encode([res[0]], convert_to_tensor=True)
  out_embeddings = paraph_model.encode(res[1], convert_to_tensor=True)
  ph = (F.normalize(in_embeddings,dim=-1)*F.normalize(out_embeddings,dim=-1)).sum(-1)
  return torch.max(util.pytorch_cos_sim(in_embeddings, out_embeddings))

In [ ]:
input_text =get_input_text(batch)
sent_emb = model.get_sentence_emb(batch)
sent_emb_stoch = sent_emb + torch.randn_like(sent_emb)
output_text = batch_beam_search_from_embedding_topk(model.model, sent_emb)
output_text = [out[0] for out in output_text]

In [ ]:
output_text_st = batch_beam_search_from_embedding_topk(model.model, sent_emb + torch.randn_like(sent_emb)*0.1)
output_text_st = [out[0] for out in output_text_st]

In [ ]:
mask = torch.rand_like(batch['input_ids'], dtype=float)< 0.15

In [ ]:
batch_masked = batch
batch_masked['input_ids'] = torch.where(mask, 5, batch['input_ids'])

In [ ]:
input_text, output_text, output_text_st

(['lumentum says halting all huawei shipments, cuts quarterly forecast',
  'johnson ’ s brexit would devastate business – the cbi must be hoping that he ’ s lying',
  "german court rules against volkswagen in'dieselgate'scandal",
  'equity trading strength boosts profits at morgan stanley, goldman',
  'switch energy providers now to escape price rises',
  "china president xi says goal of belt and road is advance'win - win cooperation '",
  'fears minimum wage rises will put more jobs at risk of automation',
  'eni cuts 2020, 2021 capex to mitigate coronavirus hit',
  "financier doubts add to boeing's max headaches",
  'malaysia regulator to probe if airasia broke rules in airbus deals'],
 ['nokia says halting forecast, halting sales for allies',
  'brexit wouldn ’ t be scrapped of the brexit ’ s business',
  "german court rules against volkswagen's diesel scandal'in diesel scandal",
  'goldman sachs profit surges in goldman sachs, trading units',
  'energy prices to step up to price hi

In [ ]:
from copy import deepcopy
def get_bert_score(model, batch, stoch=False, mask=None):
  input_text =get_input_text(batch)
  if mask is not None:
    batch_masked = deepcopy(batch)
    batch_masked['input_ids'] = torch.where(mask, 5, batch_masked['input_ids'])
    sent_emb = model.get_sentence_emb(batch_masked)
  else:
    sent_emb = model.get_sentence_emb(batch)
  if stoch:
    sent_emb = sent_emb + torch.randn_like(sent_emb)*0.1
  output_text = batch_beam_search_from_embedding_topk(model.model, sent_emb)
  output_text = [out[0] for out in output_text]
  P, R, F1 = score(output_text, input_text, lang='en', model_type='microsoft/deberta-xlarge-mnli')
  return F1.mean().item()

In [ ]:
in_embeddings = paraph_model.encode(input_text, convert_to_tensor=True)
out_embeddings = paraph_model.encode(output_text, convert_to_tensor=True)
torch.diagonal(util.pytorch_cos_sim(in_embeddings, out_embeddings))

tensor([0.5184, 0.6990, 0.9249, 0.5884, 0.4646, 0.4785, 0.7336, 0.7729, 0.4123,
        0.6308], device='cuda:0')

In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, max_length=100, sent_dim=2560).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)
#checkpoint_path = next(Path(f"/content/tb_logs/lightning_logs/version_1/checkpoints/").glob("*.ckpt"))
#model.load_state_dict(torch.load(checkpoint_path, weights_only=True)["state_dict"])
model.load_state_dict(torch.load('/content/drive/MyDrive/models/Model2560.pth', weights_only=True))

<All keys matched successfully>

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          ph_sim           │    0.5877981781959534     │
│       val_accuracy        │    0.5029309988021851     │
│         val_loss          │     2.706348180770874     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 2.706348180770874,
  'val_accuracy': 0.5029309988021851,
  'ph_sim': 0.5877981781959534}]

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          top_10           │     0.770786702632904     │
│          top_100          │    0.8681411147117615     │
│         top_1000          │     0.915876030921936     │
│          top_15           │    0.7930448651313782     │
│          top_25           │    0.8181850910186768     │
│          top_250          │    0.8911117315292358     │
│           top_3           │    0.6842784881591797     │
│           top_5           │    0.7260658740997314     │
│          top_50           │    0.8451084494590759     │
│          top_500          │    0.9049596190452576     │
│       val_accuracy        │    0.5527811646461487     │
└───────────────────────────┴───────────────────────────┘

[{'val_accuracy': 0.5527811646461487,
  'top_3': 0.6842784881591797,
  'top_5': 0.7260658740997314,
  'top_10': 0.770786702632904,
  'top_15': 0.7930448651313782,
  'top_25': 0.8181850910186768,
  'top_50': 0.8451084494590759,
  'top_100': 0.8681411147117615,
  'top_250': 0.8911117315292358,
  'top_500': 0.9049596190452576,
  'top_1000': 0.915876030921936}]

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          ph_sim           │    0.6135064363479614     │
│     val_BERTScore F1:     │    0.6761509776115417     │
│       val_accuracy        │    0.5528762340545654     │
│         val_loss          │    2.3156797885894775     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 2.3156797885894775,
  'val_accuracy': 0.5528762340545654,
  'ph_sim': 0.6135064363479614,
  'val_BERTScore F1:': 0.6761509776115417}]

In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, sent_dim=768,max_length=1000).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)
#checkpoint_path = next(Path(f"/content/tb_logs/lightning_logs/version_1/checkpoints/").glob("*.ckpt"))
#model.load_state_dict(torch.load(checkpoint_path, weights_only=True)["state_dict"])
model.load_state_dict(torch.load('/content/drive/MyDrive/models/Model768.pth', weights_only=True))

<All keys matched successfully>

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          top_10           │    0.7300444841384888     │
│          top_100          │    0.8432500958442688     │
│         top_1000          │    0.9074729681015015     │
│          top_15           │    0.7545769810676575     │
│          top_25           │    0.7821733951568604     │
│          top_250          │    0.8733267188072205     │
│           top_3           │    0.6356720328330994     │
│           top_5           │    0.6802592873573303     │
│          top_50           │    0.8152665495872498     │
│          top_500          │    0.8923140168190002     │
│       val_accuracy        │    0.5029172301292419     │
└───────────────────────────┴───────────────────────────┘

[{'val_accuracy': 0.5029172301292419,
  'top_3': 0.6356720328330994,
  'top_5': 0.6802592873573303,
  'top_10': 0.7300444841384888,
  'top_15': 0.7545769810676575,
  'top_25': 0.7821733951568604,
  'top_50': 0.8152665495872498,
  'top_100': 0.8432500958442688,
  'top_250': 0.8733267188072205,
  'top_500': 0.8923140168190002,
  'top_1000': 0.9074729681015015}]

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          ph_sim           │    0.5877981781959534     │
│       val_accuracy        │    0.5029309988021851     │
│         val_loss          │     2.706348180770874     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 2.706348180770874,
  'val_accuracy': 0.5029309988021851,
  'ph_sim': 0.5877981781959534}]

In [ ]:
torch.save(model.state_dict(), "pop.pth")

# CLS ONLY

In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, sent_dim=2560).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)

last_checkpoint_path = Path("/content/tb_logs/lightning_logs/version_1/checkpoints/last.ckpt")
trainer.validate(
    model,
    test_loader,
    ckpt_path=last_checkpoint_path,
)

INFO: Restoring states from the checkpoint path at /content/tb_logs/lightning_logs/version_1/checkpoints/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/tb_logs/lightning_logs/version_1/checkpoints/last.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/tb_logs/lightning_logs/version_1/checkpoints/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/tb_logs/lightning_logs/version_1/checkpoints/last.ckpt


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_accuracy        │    0.4064328074455261     │
│         val_loss          │    3.5615947246551514     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 3.5615947246551514, 'val_accuracy': 0.4064328074455261}]

# ENCODER ONLY

In [ ]:
import math


class EncoderDecoderModel2(nn.Module):
    def __init__(self, bert_model, hidden_dim=768, num_layers=2, nhead=8, max_length=100, vocab_size=len(tokenizer),dropout=1e-3, sent_dim=1568, tokenizer=tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.bert = bert_model  # ЭНКОДЕР (BERT)
        self.bert.requires_grad_(False)
        self.hidden_dim = hidden_dim
        self.max_length = max_length
        self.vocab_size = vocab_size
        self.sent_dim = sent_dim


        self.attn_pool = AttentionPooling(hidden_dim, sent_dim)

        self.act = nn.GELU()


        self.pos_lin1 = nn.Linear(hidden_dim, hidden_dim)
        #self.pos_lin2 = nn.Linear(hidden_dim, sent_dim)

        # Декодер
        encoder_layer = nn.TransformerEncoderLayer(d_model=sent_dim, nhead=nhead,dim_feedforward=sent_dim,dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Выходной слой для предсказания токенов
        #self.fc_out = nn.Linear(sent_dim, vocab_size)
        self.factorized_output = nn.Linear(sent_dim, vocab_size)#FactorizedOutput(sent_dim, vocab_size, sent_dim//2)

        # Генерация синусоидальных позиционных эмбеддингов
        self.pos_proj2 = nn.Sequential(
            nn.Linear(hidden_dim, sent_dim),
            nn.GELU(),
            nn.LayerNorm(sent_dim),
            nn.Dropout(p=0.001)
        )

        self.register_buffer("positional_encoding", self.sinusoidal_positional_encoding(max_length, hidden_dim))

    def sinusoidal_positional_encoding(self, seq_length, hidden_dim):
        position = torch.arange(seq_length).unsqueeze(1).float()  # (T, 1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).float() * (-math.log(10000.0) / hidden_dim))

        pe = torch.zeros(seq_length, hidden_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0).to(device)  # (1, T, H)

    def get_sentence_emb(self, input_ids, attention_mask):
        encoded = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state # (B, T, H)
        pos = self.positional_encoding[:, :input_ids.size(1), :]
        encoded += self.act(self.pos_lin1(pos))
        sent_embedding = self.attn_pool(encoded, attention_mask)
        return sent_embedding


    def forward(self, input_ids, attention_mask):
        batch_size, seq_length = input_ids.shape  # (B, T)

        with torch.no_grad():
            sent_emb = self.get_sentence_emb(input_ids, attention_mask)  # (B, S)
            tgt = self.bert.embeddings.word_embeddings(input_ids)



        # Добавляем синусоидальные позиции
        src = sent_emb.unsqueeze(1).expand(batch_size, seq_length, self.sent_dim)

        pos = self.positional_encoding[:, :seq_length, :]
        src = src + self.pos_proj2(pos)  # (B, T, S)

        # ПРОГОН ЧЕРЕЗ ДЕКОДЕР



        encoder_output = self.transformer_encoder.forward(src, src_key_padding_mask=(attention_mask==0))



        # ПРОГОН ЧЕРЕЗ ВЫХОДНОЙ ЛИНЕЙНЫЙ СЛОЙ
        token_logits = self.factorized_output(encoder_output)  # (B, T, V)

        return token_logits  # (B, T, V)

    @torch.inference_mode()
    def generate_from_embedding(self, sent_emb: torch.Tensor, mask, max_len: int = 25):
        self.eval()

        B, S = sent_emb.shape

        src = sent_emb.unsqueeze(1).expand(B, max_len, S)

        pos = self.positional_encoding[:, :max_len, :]
        src = src + self.pos_proj2(pos)

        encoder_output = self.transformer_encoder.forward(src)

        token_logits = self.factorized_output(encoder_output)
        out_tokens = token_logits.argmax(dim=-1)

        return self.tokenizer.batch_decode(out_tokens , skip_special_tokens=True), token_logits

In [ ]:
encoder_decoder_model = EncoderDecoderModel2(bert_model).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)

last_checkpoint_path = Path("/content/tb_logs/lightning_logs/version_0/checkpoints/last.ckpt")
trainer.validate(
    model,
    test_loader,
    ckpt_path=last_checkpoint_path,
)

INFO: Restoring states from the checkpoint path at /content/tb_logs/lightning_logs/version_0/checkpoints/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/tb_logs/lightning_logs/version_0/checkpoints/last.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/tb_logs/lightning_logs/version_0/checkpoints/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/tb_logs/lightning_logs/version_0/checkpoints/last.ckpt


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_accuracy        │    0.3477596938610077     │
│         val_loss          │    3.3604772090911865     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 3.3604772090911865, 'val_accuracy': 0.3477596938610077}]